In [5]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from sklearn.svm import SVC
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix, ConfusionMatrixDisplay
from sklearn.model_selection import cross_val_score, cross_val_predict, StratifiedKFold
from sklearn.decomposition import PCA
from sklearn.ensemble import RandomForestClassifier

In [2]:
df_bio = pd.read_csv('4HZ_Individual_Data/final_biometric_4HZ.csv', low_memory=False)
df_grip = pd.read_csv('4HZ_Individual_Data/final_grip_4HZ.csv', low_memory=False)
df_tel = pd.read_csv('4HZ_Individual_Data/final_telemetry_4HZ.csv', low_memory=False)
df_pose = pd.read_csv('4HZ_Individual_Data/final_pose_4HZ.csv', low_memory=False)
df_fl = pd.read_csv('4HZ_Individual_Data/final_FL_4HZ.csv', low_memory=False)
df_fau = pd.read_csv('4HZ_Individual_Data/final_FAU_4HZ.csv', low_memory=False)


#### Biometric + Behavioral ####

In [3]:
df_bio_beh =pd.concat([df_bio.drop(columns=['session','user','label']), df_grip.drop(columns=['session','user','label']), df_tel.drop(columns=['session','user','label']), df_pose], axis=1)

In [4]:
df_bio_beh['DrowsinessLevel'] = df_bio_beh['label'].apply(lambda x: 0 if x < 4 else 1 if x<7 else 2)
df_bio_beh = df_bio_beh.drop(columns=['label'])

### Applying SVM for classification (Biometric + Behavioral) ###

In [ ]:
%%time
# --- Setup Cross-validation ---
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# Separate features and target
X = df_bio_beh.drop(columns=['DrowsinessLevel', 'session', 'user'])
y = df_bio_beh['DrowsinessLevel']

# Apply Min-Max normalization
scaler = MinMaxScaler()
X_normalized = scaler.fit_transform(X)

# --- PCA Feature Selection ---
# PCA to retain a specific amount of variance
pca = PCA(n_components=0.90)  
X_pca = pca.fit_transform(X_normalized)

# Print the number of components selected and the explained variance ratio
print(f"Number of principal components selected: {X_pca.shape[1]}")
print(f"Explained variance ratio per component: {pca.explained_variance_ratio_}")
print(f"Cumulative explained variance: {np.cumsum(pca.explained_variance_ratio_)}")

# --- SVM ---
svm_clf = SVC(kernel='rbf', C=0.1, gamma='auto',class_weight='balanced', cache_size=200, random_state=42)
svm_cv_scores = cross_val_score(svm_clf, X_pca, y, cv=cv, scoring='accuracy', n_jobs=-1)
print("SVM CV Accuracy: {:.2f} (+/- {:.2f})".format(svm_cv_scores.mean(), svm_cv_scores.std() * 2))
y_cv_pred = cross_val_predict(svm_clf, X_pca, y, cv=cv, n_jobs=-1)
print("SVM CV Classification Report:\n", classification_report(y, y_cv_pred, digits=4))
cm = confusion_matrix(y, y_cv_pred, normalize='true')
ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=np.unique(y)).plot(cmap='Blues', values_format=".2f")
plt.title("SVM Normalized Confusion Matrix (5-Fold CV)")
plt.show()

# --- Random Forest ---
rf_clf = RandomForestClassifier(
    n_estimators=100,
    max_depth=5,
    min_samples_leaf=10,
    random_state=42,
    n_jobs=-1
)
rf_cv_scores = cross_val_score(rf_clf, X_pca, y, cv=cv, scoring='accuracy', n_jobs=-1)
print("RF CV Accuracy: {:.2f} (+/- {:.2f})".format(rf_cv_scores.mean(), rf_cv_scores.std() * 2))
y_cv_pred = cross_val_predict(rf_clf, X_pca, y, cv=cv, n_jobs=-1)
print("RF CV Classification Report:\n", classification_report(y, y_cv_pred, digits=4))
cm = confusion_matrix(y, y_cv_pred, normalize='true')
ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=np.unique(y)).plot(cmap='Blues', values_format=".2f")
plt.title("Random Forest Normalized Confusion Matrix (5-Fold CV)")
plt.show()


#### Biometric + Facial Features ####

In [7]:
df_bio_fac =pd.concat([df_bio.drop(columns=['session','user','label']), df_fl.drop(columns=['session','user','label']), df_fau], axis=1)

In [8]:
df_bio_fac['DrowsinessLevel'] = df_bio_fac['label'].apply(lambda x: 0 if x < 4 else 1 if x<7 else 2)
df_bio_fac = df_bio_fac.drop(columns=['label'])

### Applying SVM for classification (Biometric + Facial) ###

In [ ]:
%%time
# --- Setup Cross-validation ---
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# Separate features and target
X = df_bio_fac.drop(columns=['DrowsinessLevel', 'session', 'user'])
y = df_bio_fac['DrowsinessLevel']

# Apply Min-Max normalization
scaler = MinMaxScaler()
X_normalized = scaler.fit_transform(X)

# --- PCA Feature Selection ---
# PCA to retain a specific amount of variance
pca = PCA(n_components=0.90)  
X_pca = pca.fit_transform(X_normalized)

# Print the number of components selected and the explained variance ratio
print(f"Number of principal components selected: {X_pca.shape[1]}")
print(f"Explained variance ratio per component: {pca.explained_variance_ratio_}")
print(f"Cumulative explained variance: {np.cumsum(pca.explained_variance_ratio_)}")

# --- SVM ---
svm_clf = SVC(kernel='rbf', C=0.1, gamma='auto', cache_size=200,class_weight='balanced', random_state=42) 
svm_cv_scores = cross_val_score(svm_clf, X_pca, y, cv=cv, scoring='accuracy', n_jobs=-1)
print("SVM CV Accuracy: {:.2f} (+/- {:.2f})".format(svm_cv_scores.mean(), svm_cv_scores.std() * 2))
y_cv_pred = cross_val_predict(svm_clf, X_pca, y, cv=cv, n_jobs=-1)
print("SVM CV Classification Report:\n", classification_report(y, y_cv_pred, digits=4))
cm = confusion_matrix(y, y_cv_pred, normalize='true')
ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=np.unique(y)).plot(cmap='Blues', values_format=".2f")
plt.title("SVM Normalized Confusion Matrix (5-Fold CV)")
plt.show()

# --- Random Forest ---
rf_clf = RandomForestClassifier(
    n_estimators=100,
    max_depth=5,
    min_samples_leaf=10,
    random_state=42,
    n_jobs=-1
)
rf_cv_scores = cross_val_score(rf_clf, X_pca, y, cv=cv, scoring='accuracy', n_jobs=-1)
print("RF CV Accuracy: {:.2f} (+/- {:.2f})".format(rf_cv_scores.mean(), rf_cv_scores.std() * 2))
y_cv_pred = cross_val_predict(rf_clf, X_pca, y, cv=cv, n_jobs=-1)
print("RF CV Classification Report:\n", classification_report(y, y_cv_pred, digits=4))
cm = confusion_matrix(y, y_cv_pred, normalize='true')
ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=np.unique(y)).plot(cmap='Blues', values_format=".2f")
plt.title("Random Forest Normalized Confusion Matrix (5-Fold CV)")
plt.show()


#### Biometric + Behavioral + Facial Features ####

In [10]:
df_bio_fac_beh =pd.concat([df_bio.drop(columns=['session','user','label']),df_grip.drop(columns=['session','user','label']), df_tel.drop(columns=['session','user','label']), df_fl.drop(columns=['session','user','label']), df_fau.drop(columns=['session','user','label']), df_pose], axis=1)

In [11]:
df_bio_fac_beh['DrowsinessLevel'] = df_bio_fac_beh['label'].apply(lambda x: 0 if x < 4 else 1 if x<7 else 2)
df_bio_fac_beh = df_bio_fac_beh.drop(columns=['label'])

### Applying SVM for classification (Biometric + Behavioral + Facial Features) ###

In [ ]:
%%time
# --- Setup Cross-validation ---
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# Separate features and target
X = df_bio_fac_beh.drop(columns=['DrowsinessLevel', 'session', 'user'])
y = df_bio_fac_beh['DrowsinessLevel']

# Apply Min-Max normalization
scaler = MinMaxScaler()
X_normalized = scaler.fit_transform(X)

# --- PCA Feature Selection ---
# PCA to retain a specific amount of variance
pca = PCA(n_components=0.90)  
X_pca = pca.fit_transform(X_normalized)

# Print the number of components selected and the explained variance ratio
print(f"Number of principal components selected: {X_pca.shape[1]}")
print(f"Explained variance ratio per component: {pca.explained_variance_ratio_}")
print(f"Cumulative explained variance: {np.cumsum(pca.explained_variance_ratio_)}")

# --- SVM ---
svm_clf = SVC(kernel='rbf', C=0.1, gamma='auto', cache_size=200,class_weight='balanced', random_state=42) 
svm_cv_scores = cross_val_score(svm_clf, X_pca, y, cv=cv, scoring='accuracy', n_jobs=-1)
print("SVM CV Accuracy: {:.2f} (+/- {:.2f})".format(svm_cv_scores.mean(), svm_cv_scores.std() * 2))
y_cv_pred = cross_val_predict(svm_clf, X_pca, y, cv=cv, n_jobs=-1)
print("SVM CV Classification Report:\n", classification_report(y, y_cv_pred, digits=4))
cm = confusion_matrix(y, y_cv_pred, normalize='true')
ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=np.unique(y)).plot(cmap='Blues', values_format=".2f")
plt.title("SVM Normalized Confusion Matrix (5-Fold CV)")
plt.show()

# --- Random Forest ---
rf_clf = RandomForestClassifier(
    n_estimators=100,
    max_depth=5,
    min_samples_leaf=10,
    random_state=42,
    n_jobs=-1
)
rf_cv_scores = cross_val_score(rf_clf, X_pca, y, cv=cv, scoring='accuracy', n_jobs=-1)
print("RF CV Accuracy: {:.2f} (+/- {:.2f})".format(rf_cv_scores.mean(), rf_cv_scores.std() * 2))
y_cv_pred = cross_val_predict(rf_clf, X_pca, y, cv=cv, n_jobs=-1)
print("RF CV Classification Report:\n", classification_report(y, y_cv_pred, digits=4))
cm = confusion_matrix(y, y_cv_pred, normalize='true')
ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=np.unique(y)).plot(cmap='Blues', values_format=".2f")
plt.title("Random Forest Normalized Confusion Matrix (5-Fold CV)")
plt.show()
